# GameTheory 16j — Screening et loyer d'information (C#)

**Navigation** : [<< 16d-Players-LLM](GameTheory-16d-LLM-Players-Othman-Sandholm-Python.ipynb) | [Index GameTheory](README.md) | [Capstone 21 -- Choix social >>](GameTheory-21-Capstone-Choix-Social-Python.ipynb)

**Jumeau C# (.NET Interactive)** de [GameTheory-16j-Screening-Loyer-Information-Python.ipynb](GameTheory-16j-Screening-Loyer-Information-Python.ipynb) — **parité .NET ↔ Python**.

**Le coût mesuré de l'asymétrie d'information** dans le problème classique de *screening*
à sélection adverse (Maskin–Riley, Myerson).

À la famille **16 (Mécanismes)**, à côté de 16i (Appariage stable) : un **principal**
(employeur) offre un contrat à un **agent** (salarié) dont il **ne connaît pas le type**.
On mesure trois choses :

1. le **loyer d'information** que le principal doit verser au type haut pour que les deux
   types s'autosélectionnent honnêtement ;
2. la **distorsion** du contrat du type bas (on le fait travailler *moins* que l'optimum) —
   le type haut, lui, reste à son **premier optimum** (théorème de non-distorsion au
   sommet) ;
3. le **coût total de l'asymétrie** = surplus avec information complète − surplus avec
   screening, et sa **décomposition** en loyer + perte d'efficacité.

Toutes les valeurs sont **calculées** dans le carnet en **BCL .NET** (sans NumPy), pas
énoncées. Le jumeau C# pousse en outre le **contrôle réel** : il calcule les cas limites
(§8) que le jumeau Python laisse en exercice.


## 1. Le modèle

Un salarié est de l'un de deux **types** (productivités) :

- type **bas** $\theta_L = 1$, type **haut** $\theta_H = 2$ ;
- probabilité $p$ que le salarié soit du type haut.

Le **principal** propose un contrat $(a, t)$ : un niveau d'effort $a$ et une rémunération $t$.

| | Formule | Rôle |
|---|---|---|
| Production | $V(\theta, a) = \theta \, a$ | ce que l'effort $a$ d'un type $\theta$ rapporte au **principal** |
| Coût d'effort | $C(\theta, a) = \dfrac{a^2}{2\theta}$ | ce que l'effort $a$ coûte à **l'agent** de type $\theta$ |

Le type haut trouve l'effort **moins coûteux** ($C$ décroît en $\theta$) : c'est la condition
de **croisement simple** (single-crossing) qui rend le screening possible. **L'agent** a une
utilité $U(\theta) = t - C(\theta, a)$ ; **le principal** un surplus $S(\theta) = V(\theta, a) - t$.

> Le type est **non observable** par le principal : c'est l'asymétrie d'information.


In [1]:
using System;
using System.Linq;
using System.Text;
using System.Collections.Generic;

// Production et coût d'effort (BCL, sans NumPy).
static double V(double th, double a) => th * a;
static double C(double th, double a) => a * a / (2.0 * th);

// Types et probabilité.
double theta_L = 1.0;
double theta_H = 2.0;
double p_H     = 0.5;   // probabilité que le salarié soit du type haut

Console.WriteLine($"theta_L = {theta_L}, theta_H = {theta_H}, p_H = {p_H}");
Console.WriteLine($"coût d'effort du type haut  C(2, a=2) = {C(theta_H, 2.0)}");
Console.WriteLine($"coût d'effort du type bas   C(1, a=2) = {C(theta_L, 2.0)}");
Console.WriteLine("(le type haut trouve l'effort moins cher : croisement simple)");


The below script needs to be able to find the current output cell; this is an easy method to get it.

theta_L = 1, theta_H = 2, p_H = 0,5


coût d'effort du type haut  C(2, a=2) = 1


coût d'effort du type bas   C(1, a=2) = 2


(le type haut trouve l'effort moins cher : croisement simple)


### Lecture

À effort égal ($a=2$), le type haut paie un coût $1$ là où le type bas en paie $2$.
C'est cette **asymétrie de coût** que le mécanisme va exploiter : en proposant le bon
contrat, on fait en sorte que chaque type *choisisse de lui-même* le sien.


## 2. Premier optimum — le principal connaît le type (benchmark)

Sans asymétrie, le **principal** fixe pour **chaque** type l'effort qui maximise son surplus
$S(\theta) = \theta a - a^2/(2\theta)$, et paie juste le coût de l'agent (**aucun
loyer d'information** : la **contrainte de participation de l'agent** est liante, $U=0$).

Le maximum en $a$ donne $a^*(\theta) = \theta^2$ (dérivée nulle : $\theta - a/\theta = 0$).


In [2]:
// Premier optimum (information complète) : le principal voit le type.
double a_L_1F = theta_L * theta_L;
double a_H_1F = theta_H * theta_H;
double t_L_1F = C(theta_L, a_L_1F);          // loyer = coût  ->  loyer d'information = 0
double t_H_1F = C(theta_H, a_H_1F);
double S_L_1F = V(theta_L, a_L_1F) - t_L_1F;
double S_H_1F = V(theta_H, a_H_1F) - t_H_1F;

Console.WriteLine("PREMIER OPTIMUM (le principal voit le type) :");
Console.WriteLine($"  type bas : a* = {a_L_1F:F2},  t = {t_L_1F:F3},  surplus = {S_L_1F:F3}");
Console.WriteLine($"  type haut: a* = {a_H_1F:F2},  t = {t_H_1F:F3},  surplus = {S_H_1F:F3}");
Console.WriteLine("  (aucun loyer d'information : l'agent est payé à juste son coût)");


PREMIER OPTIMUM (le principal voit le type) :


  type bas : a* = 1,00,  t = 0,500,  surplus = 0,500


  type haut: a* = 4,00,  t = 4,000,  surplus = 4,000


  (aucun loyer d'information : l'agent est payé à juste son coût)


### Lecture

Avec information complète, le type haut travaille à $a^* = 4$ et le principal en retire
un surplus de **4.00**. C'est le **plafond** : tout ce qui manque plus bas est dû à
l'asymétrie. On garde ce chiffre en tête.


## 3. Second optimum — le principal ne voit pas le type (screening)

Le **principal** propose un **menu** de deux contrats. Les deux contraintes d'un menu
séparateur optimal :

- **Participation** du type bas liante : $U_L = 0$ (on ne verse pas de loyer au type bas) ;
- **Auto-sélection du type haut** liante : le type haut ne doit pas préférer le contrat
  du type bas — c'est cette contrainte qui crée le loyer.

Le résultat classique (croisement simple) est le **théorème de non-distorsion au
sommet** : **le type haut reçoit l'effort du premier optimum** ($a_H = \theta_H^2$,
non déformé), et **le type bas voit son effort abaissé** ($a_L < \theta_L^2$). Le loyer
d'information du type haut est alors une **fonction du contrat du type bas** :
$$U_H = C(\theta_L, a_L) - C(\theta_H, a_L) .$$

En fixant $a_H = \theta_H^2$ et en exprimant le surplus espéré du principal, son problème
se réduit au choix unique de $a_L$ :
$$\max_{a_L}\; p_H\,S_H(a_L) + (1-p_H)\,S_L(a_L)
\quad\Rightarrow\quad
a_L^{2B} = \frac{(1-p_H)\,\theta_L}{p_H\left(\frac{1}{\theta_L}-\frac{1}{\theta_H}\right) + \frac{1-p_H}{\theta_L}} .$$


In [3]:
// Second optimum (screening) — théorème de non-distorsion au sommet.
// Le type HAUT n'est PAS déformé : il travaille à son premier optimum.
double a_H_2B = theta_H * theta_H;   // type haut : NON déformé (= 1er optimum)
// Le type BAS est déformé : a_L résout le FOC du surplus espéré du principal.
double a_L_2B = ((1 - p_H) * theta_L)
               / (p_H * (1.0/theta_L - 1.0/theta_H) + (1 - p_H)/theta_L);

// Loyer d'information du type haut = fonction du contrat du type BAS.
double U_H = C(theta_L, a_L_2B) - C(theta_H, a_L_2B);

double t_L_2B = C(theta_L, a_L_2B);            // U_L = 0 (participation liante)
double t_H_2B = C(theta_H, a_H_2B) + U_H;      // t_H = coût haut + loyer (auto-sélection liante)
double S_L_2B = V(theta_L, a_L_2B) - t_L_2B;
double S_H_2B = V(theta_H, a_H_2B) - t_H_2B;

Console.WriteLine("SECOND OPTIMUM (screening, le type est caché) :");
Console.WriteLine($"  type haut : a = {a_H_2B:F2}  (NON déformé = 1er optimum),  surplus = {S_H_2B:F3}");
Console.WriteLine($"  type bas  : a = {a_L_2B:F3}  (abaissé vs {a_L_1F:F2}),   surplus = {S_L_2B:F3}");
Console.WriteLine($"  LOYER D'INFORMATION du type haut  U_H = {U_H:F3}  (= fonction de a_L)");
Console.WriteLine($"  rémunération type haut t_H = {t_H_2B:F3}  (= coût {C(theta_H, a_H_2B):F3} + loyer {U_H:F3})");


SECOND OPTIMUM (screening, le type est caché) :


  type haut : a = 4,00  (NON déformé = 1er optimum),  surplus = 3,889


  type bas  : a = 0,667  (abaissé vs 1,00),   surplus = 0,444


  LOYER D'INFORMATION du type haut  U_H = 0,111  (= fonction de a_L)


  rémunération type haut t_H = 4,111  (= coût 4,000 + loyer 0,111)


### Lecture

Contre-intuitif : c'est le **type bas** qui est déformé (de $a^* = 1$ à $a_L = 0.67$),
pas le type haut. Le type haut travaille à son **premier optimum** $a_H = 4$ — c'est le
**théorème de non-distorsion au sommet**. En échange de ce contrat non déformé, le
**principal** lui verse un **loyer d'information** $U_H = 0.11$ : c'est le prix pour qu'il
ne se fasse pas passer pour le type bas. Ce loyer est une fonction du contrat du type
**bas** — c'est la distorsion de $a_L$ qui le finance.


## 4. Le coût de l'asymétrie, et sa décomposition

Le **coût de l'asymétrie d'information** est la perte de surplus du **principal** par
rapport au premier optimum (§2), et il touche **les deux types** :

- le **type haut** perd son **loyer d'information** $U_H$ (son effort n'est PAS déformé) ;
- le **type bas** subit la **distorsion** de son effort ($a_L$ abaissé de $1$ à $0.67$).

C'est l'image miroir de l'intuition naïve : dans le screening à croisement simple,
**toute la distorsion est en bas, tout le loyer est en haut**.


In [4]:
// Coût de l'asymétrie + décomposition (les deux types).
// Type haut : tout le coût est le loyer (son effort n'est pas déformé).
double cost_H    = S_H_1F - S_H_2B;
double rent_H    = U_H;
double distort_H = 0.0;   // a_H non déformé -> perte d'efficacité nulle
// Type bas : tout le coût est la distorsion (pas de loyer, U_L = 0).
double cost_L    = S_L_1F - S_L_2B;
double distort_L = S_L_1F - (V(theta_L, a_L_2B) - C(theta_L, a_L_2B));
double rent_L    = 0.0;   // U_L = 0

Console.WriteLine("COÛT DE L'ASYMÉTRIE (par salarié, par type) :");
Console.WriteLine($"  type haut : coût = {cost_H:F3}  = loyer {rent_H:F3} + distorsion {distort_H:F3}");
Console.WriteLine($"  type bas  : coût = {cost_L:F3}  = loyer {rent_L:F3} + distorsion {distort_L:F3}");
if (Math.Abs(rent_H + distort_H - cost_H) > 1e-9) throw new Exception("décomp. type haut incohérente");
if (Math.Abs(rent_L + distort_L - cost_L) > 1e-9) throw new Exception("décomp. type bas incohérente");

// Coût espéré + décomposition espérée.
double cost_expected    = p_H * cost_H + (1 - p_H) * cost_L;
double rent_expected    = p_H * rent_H + (1 - p_H) * rent_L;
double distort_expected = p_H * distort_H + (1 - p_H) * distort_L;
Console.WriteLine($"  coût ESPÉRÉ (p_H = {p_H}) = {cost_expected:F3}");
Console.WriteLine($"    dont loyer d'info espéré   = {rent_expected:F3}   ({100*rent_expected/cost_expected:0} %)");
Console.WriteLine($"    dont distorsion espérée    = {distort_expected:F3}   ({100*distort_expected/cost_expected:0} %)");


COÛT DE L'ASYMÉTRIE (par salarié, par type) :


  type haut : coût = 0,111  = loyer 0,111 + distorsion 0,000


  type bas  : coût = 0,056  = loyer 0,000 + distorsion 0,056


  coût ESPÉRÉ (p_H = 0,5) = 0,083


    dont loyer d'info espéré   = 0,056   (67 %)


    dont distorsion espérée    = 0,028   (33 %)


### Lecture — le résultat central

Avec $\theta_L=1, \theta_H=2, p_H=0.5$, le coût **espéré** de l'asymétrie est
$1/12 \approx 0.083$. Il se décompose en **loyer d'information** ($1/18$, porté à 100 %
par le type haut) + **distorsion** ($1/36$, portée à 100 % par le type bas). Le partage
n'est PAS 50/50 entre loyer et distorsion — c'est une conséquence des paramètres.
L'essentiel : **le type haut paie en loyer sans être déformé ; le type bas est déformé
sans loyer**. C'est la signature du screening à croisement simple.


## 5. Sensibilité — le coût croît avec l'écart de type

Que se passe-t-il quand les deux types sont **plus différents** ? On fixe
$\theta_L = 1$ et on fait varier $\theta_H$, en recalculant le coût **espéré** de
l'asymétrie (loyer du type haut + distorsion du type bas) pour chaque écart.


In [5]:
// Sensibilité : coût d'asymétrie espéré en fonction de l'écart de type.
Console.WriteLine("  theta_H |    a_H |   a_L 2B |   loyer H |   dist. L |   coût esp.");
Console.WriteLine(new string('-', 60));
foreach (double thH in new double[] { 1.25, 1.5, 2.0, 2.5, 3.0 })
{
    double aH = thH * thH;   // type haut : 1er optimum (non déformé)
    double aL = ((1 - p_H) * theta_L) / (p_H * (1.0/theta_L - 1.0/thH) + (1 - p_H)/theta_L);
    double rentH = C(theta_L, aL) - C(thH, aL);   // loyer du type haut = f(a_L)
    double S1F_H = V(thH, aH) - C(thH, aH);
    double S2B_H = V(thH, aH) - (C(thH, aH) + rentH);
    double costH = S1F_H - S2B_H;   // = rentH (a_H non déformé)
    double aL1F  = theta_L * theta_L;
    double S1F_L = V(theta_L, aL1F) - C(theta_L, aL1F);
    double S2B_L = V(theta_L, aL) - C(theta_L, aL);
    double distL = S1F_L - S2B_L;   // distorsion du type bas
    double cost_exp = p_H * costH + (1 - p_H) * distL;
    Console.WriteLine($"{thH,8:F2} | {aH,6:F2} | {aL,8:F3} | {rentH,9:F3} | {distL,9:F3} | {cost_exp,10:F3}");
}


  theta_H |    a_H |   a_L 2B |   loyer H |   dist. L |   coût esp.


------------------------------------------------------------


    1,25 |   1,56 |    0,833 |     0,069 |     0,014 |      0,042


    1,50 |   2,25 |    0,750 |     0,094 |     0,031 |      0,063


    2,00 |   4,00 |    0,667 |     0,111 |     0,056 |      0,083


    2,50 |   6,25 |    0,625 |     0,117 |     0,070 |      0,094


    3,00 |   9,00 |    0,600 |     0,120 |     0,080 |      0,100


### Lecture

Le coût de l'asymétrie **croît de façon concave** avec l'écart de type : $\approx 0.04$
pour $\theta_H=1.25$, $0.083$ pour $\theta_H=2$, $\approx 0.10$ pour $\theta_H=3$. Plus
les types sont différents, plus le loyer à verser au type haut (et la distorsion du type
bas) sont gros — le résultat qu'on attendait, **mesuré** plutôt que supposé.


## 6. Exercices

### Exercice 1 — Trois types
Étendez le calcul à trois types $\theta \in \{1, 1.5, 2\}$ (probabilités
$1/3$ chacun). Calculez le **loyer d'information** du type du milieu.

*Indice : dans un menu à $n$ types, seul le type le plus bas a un loyer nul ; chaque type
supérieur cumule les écarts de coût avec le type juste en dessous.*

> **Le jumeau C# calcule ce cas en §8** (contrôle réel) : la valeur du loyer du type du
> milieu y est obtenue par maximisation numérique du surplus espéré.


In [6]:
// TODO étudiant — loyer d'information du type du milieu (3 types θ ∈ {1, 1.5, 2}, 1/3 chacun).
double theta_mid = 1.5;
double? loyer_type_milieu = null;   // TODO étudiant
Console.WriteLine("Exercice à compléter : loyer du type milieu = " +
    (loyer_type_milieu.HasValue ? loyer_type_milieu.Value.ToString("F4") : "<à calculer>"));


Exercice à compléter : loyer du type milieu = <à calculer>


### Exercice 2 — Vérifier l'auto-sélection
Vérifiez numériquement que, au second optimum, le **type haut** ne préfère pas le contrat
du type bas (c'est cette contrainte d'auto-sélection, **liante**, qui fixe le loyer).

*Indice : comparez l'utilité du type haut s'il prenait le contrat bas, $t_L - C(\theta_H, a_L)$,
à son utilité sur son propre contrat.*


In [7]:
// TODO étudiant — le type haut préfère-t-il le contrat bas ?
double? utilite_haut_si_contrat_bas = null;   // TODO étudiant : t_L_2B - C(theta_H, a_L_2B)
bool? auto_selection_respectee = null;        // TODO étudiant : true si <= utilité sur son propre contrat
Console.WriteLine("Exercice à compléter : utilité du haut sur contrat bas = " +
    (utilite_haut_si_contrat_bas.HasValue ? utilite_haut_si_contrat_bas.Value.ToString("F4") : "<à calculer>") +
    "  ->  auto-sélection respectée ? " +
    (auto_selection_respectee.HasValue ? auto_selection_respectee.Value.ToString().ToLower() : "<à calculer>"));


Exercice à compléter : utilité du haut sur contrat bas = <à calculer>  ->  auto-sélection respectée ? <à calculer>


### Exercice 3 — Égalité loyer / distorsion
Pour $\theta_L = 1$, trouvez la valeur de $\theta_H$ pour laquelle le **loyer espéré**
(type haut) et la **distorsion espérée** (type bas) sont **égaux**.

> **Correction du jumeau C#** : à $p_H = 1/2$, cette égalité **n'a pas de solution finie
> non dégénérée** — le loyer espéré reste strictement supérieur à la distorsion espérée
> pour tout $\theta_H$ fini, et les deux ne s'égalent qu'à la **limite** $\theta_H \to
> \infty$. La question n'est donc bien posée que pour un **autre** prior.
>
> **Réénoncé** : choisissez $p_H \neq 1/2$ pour lequel un **croisement fini** existe, et
> localisez-le. (§8 calcule le cas limite à $p_H=1/2$ et le croisement fini à
> $p_H = 0.7$.)

*Indice : à $p_H = 0.7$, le croisement est entre $\theta_H = 1.25$ et $\theta_H = 2$ —
vérifiez-le, ne l'affirmez pas.*


In [8]:
// TODO étudiant — à p_H = 0.5, il n'existe PAS de θ_H fini où loyer espéré == distorsion espérée
// (l'égalité n'est atteinte qu'à la limite θ_H → ∞). Choisissez un autre p_H pour lequel un
// croisement fini existe, et localisez-le.
double? p_autre = null;            // TODO étudiant : un p_H (≠ 0.5) avec croisement fini
double? theta_loyer_dist = null;   // TODO étudiant
Console.WriteLine("Exercice à compléter : p_H avec croisement fini = " +
    (p_autre.HasValue ? p_autre.Value.ToString("F2") : "<à choisir>") +
    "  ;  theta_H de l'égalité = " +
    (theta_loyer_dist.HasValue ? theta_loyer_dist.Value.ToString("F3") : "<à calculer>"));


Exercice à compléter : p_H avec croisement fini = <à choisir>  ;  theta_H de l'égalité = <à calculer>


## 7. Ce que ce carnet établit

- Le **screening** n'est pas sans coût : le type informé (type haut) paie un **loyer
  d'information**, et le type bas est **déformé** (travaille moins que l'optimum). Le
  type haut, lui, reste à son premier optimum — **non-distorsion au sommet**.
- Ce coût se **décompose** de façon exacte en loyer (type haut) + distorsion (type bas) —
  ici loyer $1/18$ + distorsion $1/36$ = coût espéré $1/12$, ce qui n'est **pas** un
  partage 50/50 mais une conséquence des paramètres choisis.
- Le coût **croît concave** avec l'écart de type : l'asymétrie d'information coûte d'autant
  plus cher que les types sont différents.

C'est le fondement des mécanismes de §16 (conception de mécanismes) : tout contrat qu'on
propose à un agent informé doit payer ce loyer, et le concevoir, c'est minimiser ce coût.


## 8. Contrôle réel du jumeau (le C# pousse la vérification)

Les trois exercices restent des **défis non résolus** pour l'étudiant (parité avec le
jumeau Python). Le jumeau C# va plus loin : il **calcule réellement** les cas limites,
pour éprouver les énoncés plutôt que de les laisser en l'air. Deux contrôles :

1. **Le cas original à trois types équiprobables** (Exercice 1) — le loyer du type du
   milieu, obtenu par maximisation numérique du surplus espéré ;
2. **L'égalité loyer/distorsion** (Exercice 3) — on mesure qu'à $p_H = 1/2$ le
   croisement n'existe **qu'à la limite** $\theta_H \to \infty$, et on le localise pour
   une **autre** probabilité ($p_H = 0.7$) où il est fini.


In [9]:
// Contrôle réel : cas à 3 types équiprobables (θ = {1, 1.5, 2}, p = 1/3 chacun).
double[] th = { 1.0, 1.5, 2.0 };
double p3 = 1.0/3.0;
double a3 = th[2]*th[2];   // non-distorsion au sommet : type haut au 1er optimum
// Surplus espéré en (a1, a2) ; a3 fixe. U1 = 0 ; U2 = C(t1,a1)-C(t2,a1) ; U3 = C(t2,a2)-C(t3,a2).
double Surplus3(double a1, double a2)
{
    double U1 = 0.0;
    double U2 = C(th[0], a1) - C(th[1], a1);
    double U3 = C(th[1], a2) - C(th[2], a2);
    double S1 = V(th[0], a1) - C(th[0], a1) - U1;
    double S2 = V(th[1], a2) - C(th[1], a2) - U2;
    double S3 = V(th[2], a3) - C(th[2], a3) - U3;
    return p3*(S1 + S2 + S3);
}
// Maximisation numérique : grille fine + raffinement local.
double bestA1 = 0, bestA2 = 0, best = -1e18;
for (double a1 = 0; a1 <= 1.0001; a1 += 0.001)
    for (double a2 = 0; a2 <= 4.0001; a2 += 0.001)
    { double s = Surplus3(a1, a2); if (s > best) { best = s; bestA1 = a1; bestA2 = a2; } }
for (int r = 0; r < 3; r++)
{
    double step = 0.001 / Math.Pow(10, r + 1);
    for (double a1 = bestA1 - 0.001; a1 <= bestA1 + 0.0011; a1 += step)
        for (double a2 = bestA2 - 0.001; a2 <= bestA2 + 0.0011; a2 += step)
        { double s = Surplus3(a1, a2); if (s > best) { best = s; bestA1 = a1; bestA2 = a2; } }
}
double U_mid = C(th[0], bestA1) - C(th[1], bestA1);   // loyer du type du milieu
Console.WriteLine("3 types {1, 1.5, 2}, p = 1/3 (contrôle réel) :");
Console.WriteLine($"  a1 = {bestA1:F4},  a2 = {bestA2:F4},  a3 = {a3:F2} (non déformé)");
Console.WriteLine($"  LOYER d'information du type du milieu  U2 = {U_mid:F4}  (= C(1,a1) - C(1.5,a1))");
Console.WriteLine($"  surplus espéré (contrôlé) = {best:F4}");


3 types {1, 1.5, 2}, p = 1/3 (contrôle réel) :


  a1 = 0,7500,  a2 = 1,8000,  a3 = 4,00 (non déformé)


  LOYER d'information du type du milieu  U2 = 0,0937  (= C(1,a1) - C(1.5,a1))


  surplus espéré (contrôlé) = 1,9083


### Lecture — le cas à trois types, contrôlé

La maximisation numérique converge vers $a_1 = 0.75$, $a_2 = 1.8$ (le type haut reste
à $a_3 = 4$), d'où le **loyer du type du milieu** $U_2 = C(1, 0.75) - C(1.5, 0.75)
\approx 0.094$. Le cas de l'Exercice 1 n'est plus affirmé : il est **calculé** — c'est le
« cas original à trois types équiprobables » contrôlé réellement.


In [10]:
// Contrôle réel : égalité loyer espéré / distorsion espérée.
// gap(thH, pH) = loyer espéré - distorsion espérée.
double Gap(double thH, double pH)
{
    double aL = ((1 - pH) * theta_L) / (pH * (1.0/theta_L - 1.0/thH) + (1 - pH)/theta_L);
    double rent = pH * (C(theta_L, aL) - C(thH, aL));
    double aL1F = theta_L * theta_L;
    double S1F_L = V(theta_L, aL1F) - C(theta_L, aL1F);
    double S2B_L = V(theta_L, aL) - C(theta_L, aL);
    double dist = (1 - pH) * (S1F_L - S2B_L);
    return rent - dist;
}
Console.WriteLine("p_H = 0.5 : gap (loyer - distorsion) par theta_H (restant > 0 = pas de croisement fini) :");
foreach (double thH in new double[] { 1.5, 2.0, 3.0, 10.0, 100.0 })
    Console.WriteLine($"  theta_H = {thH,6:F1}  ->  gap = {Gap(thH, 0.5):+0.0000;0.0000}");
// Autre probabilité : p_H = 0.7 -> un croisement fini existe ; on le localise par interpolation.
double pH2 = 0.7;
double? cross = null;
double prevTh = 1.25, prevGap = Gap(1.25, pH2);
for (double thH = 1.251; thH <= 3.0001; thH += 0.001)
{
    double g = Gap(thH, pH2);
    if (prevGap * g <= 0 && prevGap != 0)
    { cross = prevTh + (0 - prevGap) * (thH - prevTh) / (g - prevGap); break; }
    prevTh = thH; prevGap = g;
}
Console.WriteLine($"p_H = {pH2} : croisement fini localisé à theta_H ≈ {cross?.ToString("F3") ?? "non trouvé"}");


p_H = 0.5 : gap (loyer - distorsion) par theta_H (restant > 0 = pas de croisement fini) :


  theta_H =    1,5  ->  gap = +0,0313


  theta_H =    2,0  ->  gap = +0,0278


  theta_H =    3,0  ->  gap = +0,0200


  theta_H =   10,0  ->  gap = +0,0062


  theta_H =  100,0  ->  gap = +0,0006


p_H = 0,7 : croisement fini localisé à theta_H ≈ 1,750


### Lecture — l'égalité loyer/distorsion, mesurée

À $p_H = 1/2$, le balayage montre le gap (loyer − distorsion) **strictement positif** pour
tout $\theta_H$ fini, tendant vers $0$ seulement quand $\theta_H \to \infty$ : **pas de
solution finie non dégénérée**, exactement ce que l'énoncé d'origine laissait sous-entendu
sans le prouver. À $p_H = 0.7$, un **croisement fini** apparaît (localisé numériquement
entre $1.25$ et $2$) : la question est alors bien posée. Le jumeau C# transforme
l'énoncé de l'exercice en fait **mesuré**.
